In [1]:
import surprise
import pandas as pd
import numpy as np
import os
os.chdir("C:/Users/PGCP-AI/ML/MachineLearning/Cases_Rec_Sys/filmtrust")

In [2]:
rating = pd.read_csv('ratings.txt',sep=' ',names = ['uid','Iid','rating'])
rating

,uid,Iid,rating
0,1,1,2.0
1,1,2,4.0
2,1,3,3.5
3,1,4,3.0
4,1,5,4.0
...,...,...,...
35492,1508,84,3.5
35493,1508,17,4.0
35494,1508,669,1.0
35495,1508,686,2.5


In [3]:
lowest_rating = rating['rating'].min()
highest_rating = rating['rating'].max()
lowest_rating,highest_rating

(0.5, 4.0)

In [4]:
reader = surprise.Reader(rating_scale=(lowest_rating,highest_rating))
data = surprise.Dataset.load_from_df(df=rating,reader=reader)

In [5]:
similarity_options = {'name':'cosine','user_based':True}
algo = surprise.KNNBasic(sim_options=similarity_options)
output = algo.fit(data.build_full_trainset()) 

Computing the cosine similarity matrix...
Done computing similarity matrix.


In [6]:
from surprise.model_selection import GridSearchCV
from surprise.model_selection.split import KFold
param_grid ={'n_epochs' : np.arange(5, 50, 10),
            'lr_all': np.linspace(0.001, 1, 5),
            'reg_all': np.linspace(0.01, 0.8, 5)}
kfold= KFold(n_splits=5,shuffle=True,random_state=26)
# gs=GridSearchCV(surprise.KNNBasic,param_grid,measures=['rmse','mae'],cv=kfold)
gs=GridSearchCV(surprise.SVD,param_grid,measures=['rmse','mae'], joblib_verbose = 3,cv=kfold)
gs.fit(data)

[Parallel(n_jobs=1)]: Done  31 tasks      | elapsed:   12.4s
[Parallel(n_jobs=1)]: Done 127 tasks      | elapsed:   51.7s
[Parallel(n_jobs=1)]: Done 287 tasks      | elapsed:  3.2min
[Parallel(n_jobs=1)]: Done 511 tasks      | elapsed:  8.5min
[Parallel(n_jobs=1)]: Done 625 out of 625 | elapsed: 12.3min finished


In [7]:
print(gs.best_score['rmse'])
print(gs.best_params['rmse'])

0.810611494323146
{'n_epochs': 45, 'lr_all': 0.001, 'reg_all': 0.20750000000000002}


In [8]:
user = 50
u_iid = rating[rating['uid']==user]['Iid'].unique()
iids = rating['Iid'].unique()
print('List of items rated by user:',u_iid)
print('No. of items rated by user {0} : {1} '.format(user,len(u_iid)))
iids_to_predict = np.setdiff1d(iids,u_iid)
print("Items not rated by the user or those items for which the expected ratings are to be predicted",iids_to_predict)

List of items rated by user: [  8 211   3   2 219 234  12 254 250 207  11 253 236  84  10   7 233  13
   1   5   6 252 241 216 257 206   4 217   9 215 213  17 255 220 121 245
 239 251 235]
No. of items rated by user 50 : 39 
Items not rated by the user or those items for which the expected ratings are to be predicted [  14   15   16 ... 2069 2070 2071]


In [9]:
testset = [[user,iid,0.] for iid in iids_to_predict]
predictions = algo.test(testset)
exp_ratings = [ (predictions[i].iid, predictions[i].est) for i in range(0, len(predictions))]
df  = pd.DataFrame(exp_ratings, columns = ["iid", "est_rating"])
df.sort_values('est_rating',ascending=False)

,iid,est_rating
562,602,4.0
1750,1790,4.0
563,603,4.0
561,601,4.0
1753,1793,4.0
...,...,...
1480,1520,0.5
1487,1527,0.5
895,935,0.5
1514,1554,0.5


In [11]:
os.chdir("C:/Users/PGCP-AI/ML/MachineLearning/Cases_Rec_Sys/ml-100k/")

In [12]:
movies = pd.read_csv('movies_list.csv',encoding='latin-1')

In [13]:
user = 1000
u_iid = rating[rating['uid']==user]['Iid'].unique()
iids = rating['Iid'].unique()
iids_to_predict = np.setdiff1d(iids,u_iid)
testset = [[user,iid,0.] for iid in iids_to_predict]
predictions = algo.test(testset)
exp_ratings = [ (predictions[i].iid, predictions[i].est) for i in range(0, len(predictions))]
df  = pd.DataFrame(exp_ratings, columns = ["iid", "est_rating"])
df = df.merge(movies[['movie id ', ' movie title ', ' release date ']], how = 'left', left_on = 'iid', right_on = 'movie id ')
df.sort_values(by = 'est_rating', ascending = False).head(10)[['movie id ', ' movie title ', ' release date ']]

,movie id,movie title,release date
1216,1268.0,Bitter Moon (1992),01-Jan-92
1208,1260.0,Total Eclipse (1995),01-Jan-95
1724,NaN,NaN,NaN
1725,NaN,NaN,NaN
528,580.0,"Englishman Who Went Up a Hill, But Came Down a...",01-Jan-95
145,162.0,On Golden Pond (1981),01-Jan-81
1727,NaN,NaN,NaN
1728,NaN,NaN,NaN
1729,NaN,NaN,NaN
1925,NaN,NaN,NaN


In [14]:
from ipywidgets import interact
import ipywidgets as widgets
from IPython.display import display,clear_output

In [15]:
def predict_user_review(user):
    u_iid = rating[rating['uid']==user]['Iid'].unique()
    iids_to_predict = np.setdiff1d(iids,u_iid)
    testset = [[user,iid,0.] for iid in iids_to_predict]
    predictions = algo.test(testset)
    exp_ratings = [ (predictions[i].iid, predictions[i].est) for i in range(0, len(predictions))]
    df  = pd.DataFrame(exp_ratings, columns = ["iid", "est_rating"])
    df = df.merge(movies[['movie id ', ' movie title ', ' release date ']], how = 'left', left_on = 'iid', right_on = 'movie id ')
    return df.sort_values(by = 'est_rating', ascending = False).head(10)[['movie id ', ' movie title ', ' release date ']]

In [16]:
rating['uid'].min(),rating['uid'].max()
rating['Iid'].min(),rating['Iid'].max()


(1, 2071)

In [17]:
unique_users = rating['uid'].unique()

In [18]:
output = interact(predict_user_review, user = iids)
display(output)

,movie id,movie title,release date
1255,1268.0,Bitter Moon (1992),01-Jan-92
1630,1643.0,Angel Baby (1995),10-Jan-97
1654,1667.0,"Next Step, The (1995)",13-Jun-97
149,162.0,On Golden Pond (1981),01-Jan-81
1653,1666.0,Ripe (1996),02-May-97
283,296.0,"Promesse, La (1996)",16-May-97
1652,1665.0,"Brother's Kiss, A (1997)",25-Apr-97
1162,1175.0,Hugo Pool (1997),01-Jan-97
1154,1167.0,"Sum of Us, The (1994)",01-Jan-94
1146,1159.0,Stalker (1979),01-Jan-79


interactive(children=(Dropdown(description='user', options=(1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15,…

<function __main__.predict_user_review(user)>